In [1]:
!conda env list


# conda environments:
#
# * -> active
# + -> frozen
                         /home/njm12/ATMS_523/envs/era5-850
                         /home/njm12/ATMS_523/envs/xarray-climate
base                 *   /opt/conda



In [2]:
!conda run -p /home/njm12/ATMS_523/envs/xarray-climate python -m ipykernel install --user --name xarray-climate --display-name "Python (xarray-climate)"

Installed kernelspec xarray-climate in /home/njm12/.local/share/jupyter/kernels/xarray-climate



In [3]:
import sys
print(sys.executable)

/home/njm12/ATMS_523/envs/xarray-climate/bin/python


In [4]:
import pandas as pd
from astral.sun import sun
from astral import LocationInfo
from datetime import datetime
import pytz

In [5]:
tornado_csv = "/home/njm12/ATMS_596/1950-2024_actual_tornadoes.csv"

# Configuration
states_of_interest = ["MO", "IA", "IL"]
min_date = pd.to_datetime("1950-01-01")

# Automatic date parsing
tornadoes = pd.read_csv(tornado_csv, parse_dates=["date"])

# Filter by state, EF-scale, and date
tornadoes = tornadoes[
    tornadoes["st"].isin(states_of_interest) &
    tornadoes["mag"].isin([0,1,2,3,4,5]) &
    (tornadoes["date"] >= min_date)
].copy()

print("Filtered tornadoes:", tornadoes.shape)

tornadoes.head()

Filtered tornadoes: (8287, 29)


,om,yr,mo,dy,date,time,tz,st,stf,stn,...,len,wid,ns,sn,sg,f1,f2,f3,f4,fc
6,198,1950,12,2,1950-12-02,15:00:00,3,IL,17,7,...,18.8,50,1,1,1,119,117,0,0,0
7,199,1950,12,2,1950-12-02,16:00:00,3,IL,17,8,...,18.0,200,1,1,1,119,5,0,0,0
9,201,1950,12,2,1950-12-02,17:30:00,3,IL,17,9,...,9.6,50,1,1,1,157,0,0,0,0
11,5,1950,1,25,1950-01-25,19:30:00,3,MO,29,2,...,2.3,300,1,1,1,93,0,0,0,0
12,6,1950,1,25,1950-01-25,21:00:00,3,IL,17,3,...,0.1,100,1,1,1,91,0,0,0,0


In [6]:
# Ensure western longitudes are negative
tornadoes["slon"] = tornadoes["slon"].apply(lambda x: -x if x > 0 else x)

In [7]:
tornadoes["datetime"] = pd.to_datetime(tornadoes["date"].astype(str) + " " + tornadoes["time"].astype(str))

# Assign Central timezone
central = pytz.timezone("America/Chicago")
tornadoes["datetime"] = tornadoes["datetime"].dt.tz_localize(central)

In [8]:
def compute_solar_times(row):

    lat = row["slat"]
    lon = row["slon"]

    # ensure west longitude
    lon = -abs(lon)

    date = row["date"]

    try:

        location = LocationInfo(latitude=lat, longitude=lon)

        s = sun(location.observer, date=date, tzinfo=central)

        sunrise = s["sunrise"].astimezone(central)
        sunset = s["sunset"].astimezone(central)

        # Fixed here: to ensure sunset occurs AFTER sunrise
        if sunset < sunrise:
            sunset = sunset + pd.Timedelta(days=1)

        return pd.Series({
            "sunrise_local": sunrise,
            "sunset_local": sunset
        })

    except:
        return pd.Series({
            "sunrise_local": pd.NaT,
            "sunset_local": pd.NaT
        })

In [9]:
solar_times = tornadoes.apply(compute_solar_times, axis=1)

tornadoes = pd.concat([tornadoes, solar_times], axis=1)

In [10]:
df = tornadoes

In [11]:
def classify_day_night(row):

    event_time = row["datetime"]
    sunrise = row["sunrise_local"]
    sunset = row["sunset_local"]

    if pd.isnull(sunrise) or pd.isnull(sunset):
        return None

    if sunrise <= event_time <= sunset:
        return "Day"
    else:
        return "Night"

df["day_night_bin"] = df.apply(classify_day_night, axis=1)

In [12]:
# -----------------------------------------------------------
# Repair rows where solar calculation failed
# -----------------------------------------------------------

missing = df[df["day_night_bin"].isna()].copy()

print("Rows missing solar classification:", len(missing))

for idx, row in missing.iterrows():

    try:

        lat = row["slat"]
        lon = -abs(row["slon"])
        date = row["date"]

        location = LocationInfo(latitude=lat, longitude=lon)

        s = sun(location.observer, date=date)

        sunrise = s["sunrise"].astimezone(central)
        sunset = s["sunset"].astimezone(central)

        if sunset < sunrise:
            sunset = sunset + pd.Timedelta(days=1)

        df.loc[idx, "sunrise_local"] = sunrise
        df.loc[idx, "sunset_local"] = sunset

        event_time = df.loc[idx, "datetime"]

        if sunrise <= event_time <= sunset:
            df.loc[idx, "day_night_bin"] = "Day"
        else:
            df.loc[idx, "day_night_bin"] = "Night"

    except Exception as e:

        print("Still failed:", idx, e)

print("\nRemaining missing classifications:",
      df["day_night_bin"].isna().sum())

Rows missing solar classification: 0

Remaining missing classifications: 0


In [13]:
df["diurnal_sec"] = (df["sunset_local"] - df["sunrise_local"]).dt.total_seconds()

df["nocturnal_sec"] = 86400 - df["diurnal_sec"]

df["diurnal_hours"] = df["diurnal_sec"] / 3600
df["nocturnal_hours"] = df["nocturnal_sec"] / 3600

In [14]:
print(df[[
    "date",
    "time",
    "slat",
    "slon",
    "sunrise_local",
    "sunset_local",
    "day_night_bin",
    "diurnal_hours",
    "nocturnal_hours"
]].head())

         date      time   slat   slon                    sunrise_local  \
6  1950-12-02  15:00:00  38.97 -90.05 1950-12-02 07:00:56.487661-06:00   
7  1950-12-02  16:00:00  38.75 -89.67 1950-12-02 06:58:49.244195-06:00   
9  1950-12-02  17:30:00  38.17 -89.78 1950-12-02 06:57:41.974386-06:00   
11 1950-01-25  19:30:00  37.60 -90.68 1950-01-25 07:12:19.167490-06:00   
12 1950-01-25  21:00:00  41.17 -87.33 1950-01-25 07:07:14.812744-06:00   

                       sunset_local day_night_bin  diurnal_hours  \
6  1950-12-02 16:38:03.787114-06:00           Day       9.618694   
7  1950-12-02 16:37:08.674669-06:00           Day       9.638731   
9  1950-12-02 16:39:09.003928-06:00         Night       9.690842   
11 1950-01-25 17:18:20.206660-06:00         Night      10.100289   
12 1950-01-25 16:56:38.836911-06:00         Night       9.823340   

    nocturnal_hours  
6         14.381306  
7         14.361269  
9         14.309158  
11        13.899711  
12        14.176660  


In [17]:
print("\nFinal day/night counts:")
print(df["day_night_bin"].value_counts(dropna=False))

print("Total tornadoes:", len(df))


Final day/night counts:
day_night_bin
Day      6155
Night    2132
Name: count, dtype: int64
Total tornadoes: 8287


In [15]:
output_file = "/home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times.csv"

df.to_csv(output_file, index=False)

print("Saved to:", output_file)

Saved to: /home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times.csv


In [16]:
print("Negative daylight values:", (df["diurnal_hours"] < 0).sum())

print("Min daylight hours:", df["diurnal_hours"].min())
print("Max daylight hours:", df["diurnal_hours"].max())

Negative daylight values: 0
Min daylight hours: 8.960105314722222
Max daylight hours: 15.412055629166666
